# PhoWhisper-large LoRA — src/asr: 2x T4 DDP
Attach train Dataset; enable Internet and GPU T4 x2. The training subprocess uses both GPUs through torchrun/DDP.

In [ ]:
%pip install -q "transformers==4.51.3" "peft==0.15.2" "accelerate>=1.6,<2" soundfile scipy numpy gdown

In [ ]:
from pathlib import Path
import os, sys, subprocess
TRAIN_JSON = Path('/kaggle/input/datasets/n107hoangtuong/zalo-competion/train+public_test/train/train.jsonl')
AUDIO_ROOT = TRAIN_JSON.parent
WORK = Path('/kaggle/working/lifemate-asr')
PUBLIC_REPO = 'https://github.com/tuong1901/zac2026-lifemate.git'
PUBLIC_BRANCH = 'feat-eval-LLM'
if not (WORK / '.git').is_dir():
    assert not WORK.exists() or not any(WORK.iterdir()), 'Choose an empty WORK directory for git clone'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PUBLIC_BRANCH,
                    PUBLIC_REPO, str(WORK)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(WORK), 'remote', 'get-url', 'origin'], text=True).strip()
    assert remote == PUBLIC_REPO, f'Unexpected origin: {remote}'
    branch = subprocess.check_output(['git', '-C', str(WORK), 'branch', '--show-current'], text=True).strip()
    assert branch == PUBLIC_BRANCH, f'Checkout {PUBLIC_BRANCH} before updating'
    changes = subprocess.check_output(['git', '-C', str(WORK), 'status', '--porcelain'], text=True)
    if changes.strip():
        print('Local edits found; keeping them. Commit or stash them manually before pulling updates.')
        print(changes)
    else:
        subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only', 'origin', PUBLIC_BRANCH], check=True)
assert (WORK / 'src/asr/finetune.py').is_file()
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
import torch
assert torch.cuda.device_count() == 2, 'Select T4 x2 and restart session before importing torch'
assert TRAIN_JSON.is_file(), f'Check TRAIN_JSON: {TRAIN_JSON}'
assert (AUDIO_ROOT / 'audio').is_dir(), f'Check AUDIO_ROOT: {AUDIO_ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', WORK / 'src/asr')
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Training manifest:', TRAIN_JSON)
# After updating code already imported in this kernel, restart the session before importing again.
# The train/merge subprocesses always read the current files on disk.

In [ ]:
# Download YOUR model separately; it is not automatically the large student base.
import gdown, json
USER_MODEL_ROOT = Path('/kaggle/working/models/user-drive-model')
USER_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
configs = list(USER_MODEL_ROOT.rglob('config.json')) + list(USER_MODEL_ROOT.rglob('adapter_config.json'))
if not configs:
    files = gdown.download_folder(
        url=input('Paste your Drive model folder URL: ').strip(),
        output=str(USER_MODEL_ROOT), quiet=False, use_cookies=False)
    assert files, 'Drive download failed; check sharing permissions/quota'
configs = list(USER_MODEL_ROOT.rglob('config.json')) + list(USER_MODEL_ROOT.rglob('adapter_config.json'))
assert configs, 'No HF config found: inspect downloaded files; archives may need extraction'
for config in configs:
    data = json.loads(config.read_text())
    print(config)
    print({key: data.get(key) for key in ['model_type', 'd_model', 'encoder_layers',
          'decoder_layers', 'base_model_name_or_path', 'peft_type']})
# Keep MODEL=vinai/PhoWhisper-large for now. A medium checkpoint cannot initialize large.
# The other team's checkpoint URL and any extra labelled/unlabelled audio must be specified
# before generating additional training data; weights alone contain no recoverable dataset.

In [ ]:
from src.asr.training_data import collect_clips, split_clips
clips = collect_clips(TRAIN_JSON, AUDIO_ROOT)
train_clips, val_clips = split_clips(clips, val_ratio=0.1, seed=42)
assert not ({c['audio'] for c in train_clips} & {c['audio'] for c in val_clips})
print('Unique train:', len(train_clips), 'validation:', len(val_clips))
print('First training label:', train_clips[0]['text'])

In [ ]:
MODEL = 'vinai/PhoWhisper-large'
OUT = Path('/kaggle/working/asr_training')
RESUME = None
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '-m', 'src.asr.finetune', 'train',
           '--train-json', str(TRAIN_JSON), '--audio-root', str(AUDIO_ROOT),
           '--model', MODEL, '--out', str(OUT),
           '--epochs', '3', '--rank', '32', '--batch-size', '1', '--grad-accum', '16',
           '--lr', '1e-4', '--specaug', '0.05', '--val-ratio', '0.1', '--seed', '42']
if RESUME:
    command += ['--resume', str(RESUME)]
subprocess.run(command, check=True)

In [ ]:
import json
for filename in ['baseline_metrics.json', 'validation_metrics.json', 'training_summary.json']:
    print(filename, json.loads((OUT / filename).read_text()))
# WER 0.1 = 10%; keep the base model if its WER is lower.

In [ ]:
MERGED = Path('/kaggle/working/phowhisper-large-finetuned')
subprocess.run([sys.executable, '-m', 'src.asr.finetune', 'merge',
               '--adapter', str(OUT / 'best_adapter'), '--out', str(MERGED)], check=True)
assert (MERGED / 'config.json').is_file()
assert list(MERGED.glob('*.safetensors'))

In [ ]:
from src.asr.config import ASRConfig
from src.asr.transcribe import UnifiedASRTranscriber
manifest = json.loads((OUT / 'split_manifest.json').read_text())
example = manifest['validation'][0]
asr = UnifiedASRTranscriber(ASRConfig(asr_engine='phowhisper',
    model_name_or_path=str(MERGED), batch_size=1, device='cuda'))
print('Reference:', example['text'])
print('Prediction:', asr.transcribe(example['audio']))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/phowhisper_large_finetuned', 'zip', MERGED)
print('Download from Output:', archive)
print('Retain asr_training/checkpoint-N to resume training.')